# Universal GitHub Actions Control Center

Colab **Secrets** bölümüne yalnızca `GH_TOKEN` ekleyin ve notebook erişimini etkinleştirin. Sonra tek hücreyi çalıştırın. Panel Colab hücresinin içinde açılır; herkese açık Gradio bağlantısı oluşturulmaz ve ayrıca parola/e-posta doğrulaması istenmez.

**Fine-grained PAT:** `Actions: Read and write` (workflow dispatch/run silme), `Contents: Read-only` (workflow/preset keşfi), `Administration: Read and write` (repository görünürlüğü), repo metadata erişimi. Yalnızca gerekli depolara yetki verin.

**Güvenlik:** Panel yalnızca senin Colab oturumunda çalışır. `share=False` kullanıldığı için herkese açık paylaşım bağlantısı üretilmez. GitHub token'ını hücreye yazmayın; yalnızca Colab Secrets kullanın. Tüm run geçmişini silme ve Public yapma işlemleri yıkıcı/risklidir.


In [ ]:
#@title Universal GitHub Actions Control Center
#@markdown Add only GH_TOKEN to Colab Secrets, then run. A fresh UI password is generated automatically.
import sys,subprocess,importlib.util,re,json,time
def dep(m,p=None):
    if importlib.util.find_spec(m) is None: subprocess.check_call([sys.executable,"-m","pip","install","-q",p or m])
dep("gradio","gradio>=5.20,<6"); dep("requests"); dep("yaml","pyyaml")
import requests,gradio as gr,yaml,base64
from google.colab import userdata
try: TOKEN=userdata.get("GH_TOKEN")
except Exception as e: raise RuntimeError("Colab Secrets bölümüne GH_TOKEN ekleyin ve notebook erişimini etkinleştirin.") from e
if not TOKEN: raise RuntimeError("Colab Secrets bölümünde GH_TOKEN boş olamaz.")
S=requests.Session(); S.headers.update({"Accept":"application/vnd.github+json","Authorization":"Bearer "+TOKEN,"X-GitHub-Api-Version":"2022-11-28"})
API="https://api.github.com"
def repo(x):
    x=x.strip().replace("https://github.com/","").strip("/")
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+",x): raise ValueError("Repo owner/name biçiminde olmalı.")
    return x
def req(m,p,**kw):
    z=S.request(m,API+p,timeout=30,**kw)
    if z.status_code>=400: raise RuntimeError(f"GitHub {z.status_code}: {z.text[:500]}")
    return {} if z.status_code==204 or not z.content else z.json()
def info(r): return req("GET",f"/repos/{repo(r)}")
def allruns(r):
    r=repo(r); out=[]; page=1
    while True:
        a=req("GET",f"/repos/{r}/actions/runs",params={"per_page":100,"page":page}).get("workflow_runs",[])
        out+=a
        if len(a)<100: break
        page+=1
    return out
def file_text(r,path):
    item=req("GET",f"/repos/{repo(r)}/contents/{path}",params={"ref":info(r)["default_branch"]})
    return base64.b64decode(item.get("content","")).decode("utf-8")
def discover(r):
    r=repo(r); ws=[]; benches=[]; presets=[]
    for w in req("GET",f"/repos/{r}/actions/workflows",params={"per_page":100}).get("workflows",[]):
        doc={}
        try: doc=yaml.safe_load(file_text(r,w["path"])) or {}
        except: pass
        trig=doc.get("on",doc.get(True,{})); d=trig.get("workflow_dispatch",{}) if isinstance(trig,dict) else {}
        ins=d.get("inputs",{}) if isinstance(d,dict) else {}
        ws.append((f"{w['name']} · {w['path']}",w["id"],bool(d),ins,w["path"]))
        if any(k in (w["name"]+" "+w["path"]).lower() for k in ("bench","benchmark","performance","profil")): benches.append(w["path"])
    try:
        data=json.loads(file_text(r,"CMakePresets.json"))
        presets=[x["name"] for x in data.get("buildPresets",[]) if x.get("name")]
    except: pass
    return ws,benches,presets
def table(r):
    return [[x.get("id"),x.get("name"),x.get("event"),x.get("head_branch"),x.get("status"),x.get("conclusion") or "—",x.get("created_at"),x.get("html_url")] for x in allruns(r)[:100]]
def connect(r):
    i=info(r); w,b,p=discover(r); c=[x[0] for x in w]
    msg=f"**{i['full_name']}** · {'Private' if i['private'] else 'Public'} · default {i['default_branch']}\nWorkflows: {len(w)} · Benchmark workflows: {len(b)} · Build presets: {len(p)}\n\n"
    msg+="\n".join(f"- {x[4]} — {'dispatch' if x[2] else 'no dispatch'}" for x in w)
    msg+="\n\n**Benchmarks**\n"+(", ".join(b) if b else "No named benchmark workflow found.")
    msg+="\n\n**CMake presets**\n"+(", ".join(p) if p else "None found.")
    return msg,gr.update(choices=c,value=c[0] if c else None),table(r)
def inputs_for(r,label):
    x=next((x for x in discover(r)[0] if x[0]==label),None)
    return json.dumps({k:v.get("default","") for k,v in (x[3] or {}).items()},ensure_ascii=False,indent=2) if x else "{}"
def dispatch(r,ref,label,j,pub):
    r=repo(r); x=next((x for x in discover(r)[0] if x[0]==label),None)
    if not x or not x[2]: raise ValueError("Workflow dispatch desteklemiyor veya seçim geçersiz.")
    data=json.loads(j or "{}")
    if not isinstance(data,dict) or set(data)-set((x[3] or {}).keys()): raise ValueError("Input JSON nesnesi olmalı ve yalnızca tanımlı workflow inputlarını içermeli.")
    was_private=info(r)["private"]; changed=False
    try:
        if pub and was_private:
            req("PATCH",f"/repos/{r}",json={"private":False}); changed=True; time.sleep(4)
        started=time.time()
        req("POST",f"/repos/{r}/actions/workflows/{x[1]}/dispatches",json={"ref":ref.strip(),"inputs":data})
        if not (pub and was_private):
            return f"Dispatch gönderildi: {x[4]} / {ref}"
        # Keep the repository public until the dispatched run has completed.
        deadline=time.time()+3*60*60; run_id=None
        while time.time()<min(deadline,started+300) and run_id is None:
            batch=req("GET",f"/repos/{r}/actions/workflows/{x[1]}/runs",params={"event":"workflow_dispatch","branch":ref.strip(),"per_page":100}).get("workflow_runs",[])
            for candidate in batch:
                try:
                    created=candidate.get("created_at","").replace("Z","+00:00")
                    from datetime import datetime
                    if datetime.fromisoformat(created).timestamp() >= started-15:
                        run_id=candidate["id"]; break
                except: pass
            if run_id is None: time.sleep(8)
        if run_id is None: raise TimeoutError("Dispatch kabul edildi ancak 5 dakika içinde run bulunamadı; repository Private durumuna geri alınacak.")
        while time.time()<deadline:
            state=req("GET",f"/repos/{r}/actions/runs/{run_id}")
            if state.get("status")=="completed":
                return f"Run #{run_id} tamamlandı: {state.get('conclusion')}. Repository şimdi Private yapılacak."
            time.sleep(15)
        raise TimeoutError(f"Run #{run_id} üç saat içinde tamamlanmadı; repository Private yapılacak. Run GitHub Actions üzerinden izlenmeli.")
    finally:
        if changed:
            try: req("PATCH",f"/repos/{r}",json={"private":True})
            except Exception as e: raise RuntimeError("KRİTİK: Repository Private durumuna geri alınamadı. GitHub Settings > General > Danger Zone üzerinden hemen Private yapın. "+str(e))
def delete_ids(r,s):
    a=[x for x in re.split(r"[,\s]+",s or "") if x]
    if not a or len(a)>500 or any(not x.isdigit() for x in a): raise ValueError("1-500 arası sayısal run ID girin.")
    for x in a: req("DELETE",f"/repos/{repo(r)}/actions/runs/{x}")
    return f"{len(a)} run kaydı ve ilişkili log/artifact'lar silindi."
def delete_all(r,c):
    if c!="DELETE ALL RUN HISTORY": raise ValueError("Onay metni: DELETE ALL RUN HISTORY")
    a=allruns(r); done=0; errors=[]
    for x in a:
        try: req("DELETE",f"/repos/{repo(r)}/actions/runs/{x['id']}"); done+=1
        except Exception as e: errors.append(f"{x['id']}: {e}")
    return f"Silinen {done}/{len(a)} kayıt." + (" Hatalar: "+" | ".join(errors[:10]) if errors else "")
def visibility(r,v,c):
    if c!="CHANGE REPOSITORY VISIBILITY": raise ValueError("Onay metnini eksiksiz girin.")
    x=req("PATCH",f"/repos/{repo(r)}",json={"private":v=="Private"})
    return f"{x['full_name']} → {'Private' if x['private'] else 'Public'}"
css=""".gradio-container{background:radial-gradient(ellipse at 15% 0%,#28214d,#111827 48%,#080d18)!important} #hero{padding:22px;border:1px solid #414064;border-radius:18px;background:linear-gradient(110deg,#5b4bb52b,#14b8a626)} #hero h1{background:linear-gradient(90deg,#c4b5fd,#67e8f9);-webkit-background-clip:text;color:transparent} button.primary{background:linear-gradient(90deg,#6d5ce7,#0d9488)!important;border:0!important}"""
with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet",secondary_hue="teal",neutral_hue="slate"),css=css,title="Universal GitHub Actions Control Center") as app:
    gr.HTML("<div id='hero'><h1>Universal GitHub Actions Control Center</h1>Multi-project · Workflows · Benchmarks · Run history · Visibility guard<br><small>Authenticated · GitHub token is never displayed</small></div>")
    with gr.Row():
        r=gr.Textbox(label="Repository",value="Jue-dev/Tessera"); ref=gr.Textbox(label="Branch / ref",value="improve/stability-fuzzing-and-maintainability")
    with gr.Row():
        go=gr.Button("Connect & discover",variant="primary"); refresh=gr.Button("Refresh run history")
    summary=gr.Markdown()
    with gr.Tab("Workflows & Benchmarks"):
        wf=gr.Dropdown(label="Discovered workflows",choices=[]); inp=gr.Code(label="Workflow inputs (JSON)",language="json",value="{}")
        pub=gr.Checkbox(label="Temporarily switch Private → Public for dispatch, then restore Private",value=False)
        run=gr.Button("Dispatch workflow",variant="primary"); out=gr.Markdown()
        gr.Markdown("Benchmark workflows are detected from workflow names/paths; CMake build presets are discovered. Execution runs through each project's own Actions workflow, not arbitrary shell code on Colab.")
    with gr.Tab("Actions history"):
        data=gr.Dataframe(headers=["Run ID","Workflow","Event","Branch","Status","Conclusion","Created","URL"],interactive=False,wrap=True)
        ids=gr.Textbox(label="Run IDs to delete",placeholder="44716935108")
        ds=gr.Button("Delete selected runs",variant="stop"); dsout=gr.Markdown()
        gr.Markdown("DANGER: All run records, logs and associated artifacts are deleted; this cannot be undone.")
        confirm=gr.Textbox(label="Type DELETE ALL RUN HISTORY"); da=gr.Button("Delete ALL Actions history",variant="stop"); daout=gr.Markdown()
    with gr.Tab("Repository security"):
        target=gr.Radio(["Private","Public"],value="Private",label="Target visibility")
        vc=gr.Textbox(label="Type CHANGE REPOSITORY VISIBILITY"); vb=gr.Button("Change visibility",variant="stop"); vo=gr.Markdown()
        gr.Markdown("Public exposes source and commit history. Returning to Private cannot retract copies/forks made while public.")
    go.click(connect,[r],[summary,wf,data]); refresh.click(table,[r],[data]); wf.change(inputs_for,[r,wf],[inp])
    run.click(dispatch,[r,ref,wf,inp,pub],[out]); ds.click(delete_ids,[r,ids],[dsout]); da.click(delete_all,[r,confirm],[daout]); vb.click(visibility,[r,target,vc],[vo])
app.launch(share=False,inline=True,show_error=True,prevent_thread_lock=True)
print("Panel Colab hücresinin içinde açıldı. Herkese açık paylaşım bağlantısı oluşturulmadı.")



